In [290]:
# ============================================================
# IMPORTAR LIBRERÍAS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

In [291]:
# ============================================================
# CONFIGURAR RUTAS
# ============================================================
DATA_DIR = Path("../Data")
RESULTADOS_DIR = Path("../Resultado")

ruta_productos_sin_ventas = DATA_DIR / "Productos_Sin_Ventas.xlsx"

ruta_inventario = DATA_DIR / "Inventario.xlsx"

ruta_ventas_powerbi = RESULTADOS_DIR / "Ventas_Limpias_PowerBI.xlsx"

ruta_salida_inventario = RESULTADOS_DIR / "Analisis_Inventario_Cobertura.xlsx"

ruta_tabla_maestra = DATA_DIR / "Tabla_Maestra_Productos.xlsx"

In [292]:
# ============================================================
# CARGAR INVENTARIO
# ============================================================

df_inventario = pd.read_excel(ruta_inventario)

print("Dimensiones del inventario:", df_inventario.shape)

display(df_inventario.head())

Dimensiones del inventario: (6141, 3)


,Nombre en pantalla,Unidad de medida,Cantidad disponible para uso
0,[2542] ENVASE SADMAN TRANSP 105 ML PLATA CROMA...,Unidades,59987.0
1,[2699] ENVASE PM TRANSPARENTE GRIS AGRAFE 110 ...,Unidades,400.0
2,[0001] ALCOHOL EXTRA NEUTRO.,L,2240.0
3,[0005] CAJA MADERA,Unidades,3.0
4,[0006] CAJA PARA CILINDRO 1OZ NEGRA,Unidades,8106.0


In [293]:
# ============================================================
# FUNCION CONTAR VALORES NULOS POR COLUMNA
# ============================================================
def contar_nulos(df):
    """
    Recibe un DataFrame y devuelve un nuevo DataFrame con el nombre de cada columna y cuántos valores nulos tiene.
    """
    nulos_por_columna = df.isnull().sum()
    resultado = nulos_por_columna.reset_index()
    resultado.columns = ["COLUMNA", "CANTIDAD_NULL"]
    return resultado

In [294]:
# ============================================================
# VALIDAR VALORES NULOS POR COLUMNA
# ============================================================

resultado_nulos = contar_nulos(df_inventario)
display(resultado_nulos)

,COLUMNA,CANTIDAD_NULL
0,Nombre en pantalla,0
1,Unidad de medida,0
2,Cantidad disponible para uso,0


In [295]:
# ============================================================
# ELIMINAR COLUMNA UNIDAD DE MEDIDA
# ============================================================

df_inventario = df_inventario.drop(columns=["Unidad de medida"])

print("Columna 'Unidad de medida' eliminada correctamente.")
print("Dimensiones actuales del inventario:", df_inventario.shape)

display(df_inventario.head())

Columna 'Unidad de medida' eliminada correctamente.
Dimensiones actuales del inventario: (6141, 2)


,Nombre en pantalla,Cantidad disponible para uso
0,[2542] ENVASE SADMAN TRANSP 105 ML PLATA CROMA...,59987.0
1,[2699] ENVASE PM TRANSPARENTE GRIS AGRAFE 110 ...,400.0
2,[0001] ALCOHOL EXTRA NEUTRO.,2240.0
3,[0005] CAJA MADERA,3.0
4,[0006] CAJA PARA CILINDRO 1OZ NEGRA,8106.0


In [296]:
# ============================================================
# LIMPIAR INVENTARIO
# ============================================================

df_inventario = df_inventario.copy()

# Estandarizar nombres de columnas
df_inventario.columns = (
    df_inventario.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
)

display(df_inventario.head())
print(df_inventario.columns)

,NOMBRE_EN_PANTALLA,CANTIDAD_DISPONIBLE_PARA_USO
0,[2542] ENVASE SADMAN TRANSP 105 ML PLATA CROMA...,59987.0
1,[2699] ENVASE PM TRANSPARENTE GRIS AGRAFE 110 ...,400.0
2,[0001] ALCOHOL EXTRA NEUTRO.,2240.0
3,[0005] CAJA MADERA,3.0
4,[0006] CAJA PARA CILINDRO 1OZ NEGRA,8106.0


Index(['NOMBRE_EN_PANTALLA', 'CANTIDAD_DISPONIBLE_PARA_USO'], dtype='str')


In [297]:
# ============================================================
# EXTRAER CÓDIGO Y NOMBRE DEL PRODUCTO
# ============================================================

df_inventario["CODIGO_PRODUCTO"] = (
    df_inventario["NOMBRE_EN_PANTALLA"]
    .astype(str)
    .str.extract(r"\[(.*?)\]")
)

df_inventario["NOMBRE_PRODUCTO"] = (
    df_inventario["NOMBRE_EN_PANTALLA"]
    .astype(str)
    .str.replace(r"\[.*?\]", "", regex=True)
    .str.strip()
    .str.upper()
)

df_inventario["CODIGO_PRODUCTO"] = (
    df_inventario["CODIGO_PRODUCTO"]
    .astype(str)
    .str.strip()
    .str.zfill(4)
)

df_inventario = df_inventario.rename(columns={
    "CANTIDAD_DISPONIBLE_PARA_USO": "CANTIDAD_DISPONIBLE"
})

df_inventario = df_inventario[
    [
        "CODIGO_PRODUCTO",
        "NOMBRE_PRODUCTO",
        "CANTIDAD_DISPONIBLE"
    ]
]

display(df_inventario.head())

,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CANTIDAD_DISPONIBLE
0,2542,ENVASE SADMAN TRANSP 105 ML PLATA CROMADO X 54...,59987.0
1,2699,ENVASE PM TRANSPARENTE GRIS AGRAFE 110 ML X 40...,400.0
2,0001,ALCOHOL EXTRA NEUTRO.,2240.0
3,0005,CAJA MADERA,3.0
4,0006,CAJA PARA CILINDRO 1OZ NEGRA,8106.0


In [298]:
# ============================================================
# VALIDACIONES DEL INVENTARIO
# ============================================================

print("Registros del inventario:", len(df_inventario))

print("Nulos por columna:")
display(df_inventario.isnull().sum())

productos_sin_codigo = df_inventario[
    df_inventario["CODIGO_PRODUCTO"].isna() |
    (df_inventario["CODIGO_PRODUCTO"] == "") |
    (df_inventario["CODIGO_PRODUCTO"] == "NAN")
]

print("Productos sin código:", len(productos_sin_codigo))

if len(productos_sin_codigo) > 0:
    display(productos_sin_codigo.head(20))

inventario_negativo = df_inventario[df_inventario["CANTIDAD_DISPONIBLE"] < 0]

print("Productos con inventario negativo:", len(inventario_negativo))

if len(inventario_negativo) > 0:
    display(inventario_negativo.head(20))

Registros del inventario: 6141
Nulos por columna:


CODIGO_PRODUCTO        2
NOMBRE_PRODUCTO        0
CANTIDAD_DISPONIBLE    0
dtype: int64

Productos sin código: 2


,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CANTIDAD_DISPONIBLE
6139,NaN,0523,0.0
6140,NaN,CINTURA 70,0.0


Productos con inventario negativo: 0


In [299]:
# ============================================================
# ELIMINAR REGISTROS SIN CÓDIGO DE PRODUCTO
# ============================================================

registros_antes = len(df_inventario)

df_inventario = df_inventario[
    df_inventario["CODIGO_PRODUCTO"].notna() &
    (df_inventario["CODIGO_PRODUCTO"].astype(str).str.strip() != "") &
    (df_inventario["CODIGO_PRODUCTO"].astype(str).str.upper().str.strip() != "NAN")
].copy()

registros_despues = len(df_inventario)
registros_eliminados = registros_antes - registros_despues

print("Registros antes de eliminar productos sin código:", registros_antes)
print("Registros eliminados por no tener código:", registros_eliminados)
print("Registros después de la limpieza:", registros_despues)

display(df_inventario.head())

Registros antes de eliminar productos sin código: 6141
Registros eliminados por no tener código: 2
Registros después de la limpieza: 6139


,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CANTIDAD_DISPONIBLE
0,2542,ENVASE SADMAN TRANSP 105 ML PLATA CROMADO X 54...,59987.0
1,2699,ENVASE PM TRANSPARENTE GRIS AGRAFE 110 ML X 40...,400.0
2,0001,ALCOHOL EXTRA NEUTRO.,2240.0
3,0005,CAJA MADERA,3.0
4,0006,CAJA PARA CILINDRO 1OZ NEGRA,8106.0


In [300]:
# ============================================================
# VALIDAR QUE NO EXISTAN PRODUCTOS SIN CÓDIGO
# ============================================================

productos_sin_codigo = df_inventario[
    df_inventario["CODIGO_PRODUCTO"].isna() |
    (df_inventario["CODIGO_PRODUCTO"].astype(str).str.strip() == "") |
    (df_inventario["CODIGO_PRODUCTO"].astype(str).str.upper().str.strip() == "NAN")
]

print("Productos sin código después de la limpieza:", len(productos_sin_codigo))

if len(productos_sin_codigo) > 0:
    display(productos_sin_codigo)
else:
    print("Validación correcta: no quedan productos sin código.")

Productos sin código después de la limpieza: 0
Validación correcta: no quedan productos sin código.


In [301]:
# ============================================================
# CARGAR VENTAS LIMPIAS POWER BI
# ============================================================

df_ventas = pd.read_excel(
    ruta_ventas_powerbi,
    sheet_name="Ventas_Limpias"
)

print("Dimensiones ventas:", df_ventas.shape)

display(df_ventas.head())

Dimensiones ventas: (48447, 8)


,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CLASIFICACION I,CLASIFICACION II,CLASIFICACION III,CLASIFICACION IV,PERIODO,CANTIDAD_FACTURADA
0,1,ALCOHOL EXTRA NEUTRO.,ALCOHOL,ALCOHOL,ALCOHOL,ALCOHOL,ABRIL 2024,23957.0
1,940,STCK. ESC. INSP. EN CAN CAN BURLESQUE DM,STCK,NO APLICA,NO APLICA,NO APLICA,ABRIL 2024,300.0
2,939,STCK. ESC. INSP. EN CAN CAN BLING DM,STCK,NO APLICA,NO APLICA,NO APLICA,ABRIL 2024,80.0
3,1603,STCK. ESC. INSP. EN BY FIREPLACE UNISEX,STCK,NO APLICA,NO APLICA,NO APLICA,ABRIL 2024,380.0
4,1405,STCK. ESC. INSP. EN BVLGARY POUR HOMME SOIR MEN,STCK,NO APLICA,NO APLICA,NO APLICA,ABRIL 2024,80.0


In [302]:
# ============================================================
# ESTANDARIZAR CÓDIGO DE PRODUCTO EN VENTAS
# ============================================================

df_ventas["CODIGO_PRODUCTO"] = (
    df_ventas["CODIGO_PRODUCTO"]
    .astype(str)
    .str.strip()
    .str.zfill(4)
)

df_ventas["PERIODO"] = (
    df_ventas["PERIODO"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df_ventas["CANTIDAD_FACTURADA"] = pd.to_numeric(
    df_ventas["CANTIDAD_FACTURADA"],
    errors="coerce"
)

df_ventas = df_ventas[df_ventas["CANTIDAD_FACTURADA"] > 0]

display(df_ventas.head())

,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CLASIFICACION I,CLASIFICACION II,CLASIFICACION III,CLASIFICACION IV,PERIODO,CANTIDAD_FACTURADA
0,0001,ALCOHOL EXTRA NEUTRO.,ALCOHOL,ALCOHOL,ALCOHOL,ALCOHOL,ABRIL 2024,23957.0
1,0940,STCK. ESC. INSP. EN CAN CAN BURLESQUE DM,STCK,NO APLICA,NO APLICA,NO APLICA,ABRIL 2024,300.0
2,0939,STCK. ESC. INSP. EN CAN CAN BLING DM,STCK,NO APLICA,NO APLICA,NO APLICA,ABRIL 2024,80.0
3,1603,STCK. ESC. INSP. EN BY FIREPLACE UNISEX,STCK,NO APLICA,NO APLICA,NO APLICA,ABRIL 2024,380.0
4,1405,STCK. ESC. INSP. EN BVLGARY POUR HOMME SOIR MEN,STCK,NO APLICA,NO APLICA,NO APLICA,ABRIL 2024,80.0


In [303]:
resultado_nulos_ventas = contar_nulos(df_ventas)
display(resultado_nulos_ventas)

,COLUMNA,CANTIDAD_NULL
0,CODIGO_PRODUCTO,0
1,NOMBRE_PRODUCTO,0
2,CLASIFICACION I,0
3,CLASIFICACION II,0
4,CLASIFICACION III,0
5,CLASIFICACION IV,0
6,PERIODO,0
7,CANTIDAD_FACTURADA,0


In [304]:
# ============================================================
# CALCULAR VENTA MENSUAL POR PRODUCTO
# ============================================================

df_venta_mensual_producto = (
    df_ventas
    .groupby(["CODIGO_PRODUCTO", "NOMBRE_PRODUCTO", "PERIODO"], as_index=False)
    ["CANTIDAD_FACTURADA"]
    .sum()
)

display(df_venta_mensual_producto.head())

,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,PERIODO,CANTIDAD_FACTURADA
0,0001,ALCOHOL EXTRA NEUTRO.,ABRIL 2024,23957.0
1,0001,ALCOHOL EXTRA NEUTRO.,ABRIL 2025,23674.0
2,0001,ALCOHOL EXTRA NEUTRO.,ABRIL 2026,29880.0
3,0001,ALCOHOL EXTRA NEUTRO.,AGOSTO 2024,25121.0
4,0001,ALCOHOL EXTRA NEUTRO.,AGOSTO 2025,25896.0


In [305]:
display(contar_nulos(df_venta_mensual_producto))

,COLUMNA,CANTIDAD_NULL
0,CODIGO_PRODUCTO,0
1,NOMBRE_PRODUCTO,0
2,PERIODO,0
3,CANTIDAD_FACTURADA,0


In [306]:
# ============================================================
# CALCULAR PROMEDIO MENSUAL VENDIDO POR PRODUCTO
# ============================================================

df_promedio_ventas = (
    df_venta_mensual_producto
    .groupby(["CODIGO_PRODUCTO"], as_index=False)
    .agg(
        VENTA_PROMEDIO_MENSUAL=("CANTIDAD_FACTURADA", "mean"),
        VENTA_MAXIMA_MENSUAL=("CANTIDAD_FACTURADA", "max"),
        VENTA_MINIMA_MENSUAL=("CANTIDAD_FACTURADA", "min"),
        MESES_CON_VENTA=("PERIODO", "nunique"),
        TOTAL_VENDIDO=("CANTIDAD_FACTURADA", "sum")
    )
)

display(df_promedio_ventas.head())

,CODIGO_PRODUCTO,VENTA_PROMEDIO_MENSUAL,VENTA_MAXIMA_MENSUAL,VENTA_MINIMA_MENSUAL,MESES_CON_VENTA,TOTAL_VENDIDO
0,0001,24178.275862,31825.0,7438.0,29,701170.0
1,0005,2.000000,2.0,2.0,1,2.0
2,0006,2596.551724,5500.0,150.0,29,75300.0
3,0007,1630.629630,3301.0,450.0,27,44027.0
4,0008,2014.750000,3650.0,25.0,28,56413.0


In [307]:
display(contar_nulos(df_promedio_ventas))

,COLUMNA,CANTIDAD_NULL
0,CODIGO_PRODUCTO,0
1,VENTA_PROMEDIO_MENSUAL,0
2,VENTA_MAXIMA_MENSUAL,0
3,VENTA_MINIMA_MENSUAL,0
4,MESES_CON_VENTA,0
5,TOTAL_VENDIDO,0


In [308]:
# ============================================================
# PRODUCTOS CON INVENTARIO EN CERO
# ============================================================

productos_inventario_cero = df_inventario[
    df_inventario["CANTIDAD_DISPONIBLE"] == 0
].copy()

print("Productos con inventario en cero:")
print(len(productos_inventario_cero))

display(
    productos_inventario_cero[
        [
            "CODIGO_PRODUCTO",
            "NOMBRE_PRODUCTO",
            "CANTIDAD_DISPONIBLE"
        ]
    ].sort_values(
        by="NOMBRE_PRODUCTO",
        ascending=True
    )
)

Productos con inventario en cero:
3779


,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CANTIDAD_DISPONIBLE
2557,2278 -1000,A MILKSHAKE PLEASE DM 1000,0.0
2558,2278 -120,A MILKSHAKE PLEASE DM 120,0.0
2559,2278 -250,A MILKSHAKE PLEASE DM 250,0.0
2560,2278 -500,A MILKSHAKE PLEASE DM 500,0.0
3069,252 -1000,ACQUA FRESCA HM 1000,0.0
...,...,...,...
4614,434 -250,YUMEN HM 250,0.0
4615,434 -500,YUMEN HM 500,0.0
5870,699 -1000,ZAR E UNISEX 1000,0.0
5871,699 -120,ZAR E UNISEX 120,0.0


In [309]:
# ============================================================
# VALIDAR CÓDIGOS QUE ESTÁN EN INVENTARIO PERO NO EN VENTAS
# ============================================================

codigos_inventario = set(df_inventario["CODIGO_PRODUCTO"].dropna().astype(str).str.strip())
codigos_ventas = set(df_promedio_ventas["CODIGO_PRODUCTO"].dropna().astype(str).str.strip())

codigos_sin_venta = codigos_inventario - codigos_ventas

print("Productos en inventario:", len(codigos_inventario))
print("Productos con ventas:", len(codigos_ventas))
print("Productos de inventario sin venta histórica:", len(codigos_sin_venta))

display(
    df_inventario[
        df_inventario["CODIGO_PRODUCTO"].isin(codigos_sin_venta)
    ].head(30)
)

Productos en inventario: 6139
Productos con ventas: 2469
Productos de inventario sin venta histórica: 3682


,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CANTIDAD_DISPONIBLE
0,2542,ENVASE SADMAN TRANSP 105 ML PLATA CROMADO X 54...,59987.0
1,2699,ENVASE PM TRANSPARENTE GRIS AGRAFE 110 ML X 40...,400.0
18,0023,CILINDRO COLOR 30ML ROSA BABY X 180 UND,0.0
20,0026,CILINDRO COLOR 30ML AZUL BABY X 120 UND,0.0
27,0037,CILINDRO COLOR 30ML MORADO BABY X 120 UND,0.0
35,0047,CILINDRO COLOR 100 ML ROSA X 99 UND,0.0
36,0048,CILINDRO COLOR 100 ML VERDE X 99 UND,0.0
37,0049,CILINDRO TRANSP 2OZ AMARILLO X 112 UND,0.0
38,0050,CILINDRO TRANSP 2OZ AZUL CELESTE X 112 UND,0.0
39,0052,CILINDRO TRANSP 2OZ AZUL X 120 UND,0.0


In [310]:
# ============================================================
# CRUZAR INVENTARIO CON PROMEDIO DE VENTAS
# ============================================================

df_cobertura = df_inventario.merge(
    df_promedio_ventas,
    on="CODIGO_PRODUCTO",
    how="left"
)

display(df_cobertura.head())

,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CANTIDAD_DISPONIBLE,VENTA_PROMEDIO_MENSUAL,VENTA_MAXIMA_MENSUAL,VENTA_MINIMA_MENSUAL,MESES_CON_VENTA,TOTAL_VENDIDO
0,2542,ENVASE SADMAN TRANSP 105 ML PLATA CROMADO X 54...,59987.0,NaN,NaN,NaN,NaN,NaN
1,2699,ENVASE PM TRANSPARENTE GRIS AGRAFE 110 ML X 40...,400.0,NaN,NaN,NaN,NaN,NaN
2,0001,ALCOHOL EXTRA NEUTRO.,2240.0,24178.275862,31825.0,7438.0,29.0,701170.0
3,0005,CAJA MADERA,3.0,2.000000,2.0,2.0,1.0,2.0
4,0006,CAJA PARA CILINDRO 1OZ NEGRA,8106.0,2596.551724,5500.0,150.0,29.0,75300.0


In [311]:
# ============================================================
# SEPARAR PRODUCTOS CON VENTAS Y PRODUCTOS SIN VENTAS
# ============================================================

# Productos que están en inventario pero NO tienen venta histórica
df_productos_sin_ventas = df_cobertura[
    df_cobertura["VENTA_PROMEDIO_MENSUAL"].isna()
].copy()

# Productos que están en inventario y SÍ tienen venta histórica
df_productos_con_ventas = df_cobertura[
    df_cobertura["VENTA_PROMEDIO_MENSUAL"].notna()
].copy()

print("Total productos en inventario:", len(df_cobertura))
print("Productos con ventas históricas:", len(df_productos_con_ventas))
print("Productos sin ventas históricas:", len(df_productos_sin_ventas))

display(df_productos_sin_ventas.head(20))

Total productos en inventario: 6139
Productos con ventas históricas: 2457
Productos sin ventas históricas: 3682


,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CANTIDAD_DISPONIBLE,VENTA_PROMEDIO_MENSUAL,VENTA_MAXIMA_MENSUAL,VENTA_MINIMA_MENSUAL,MESES_CON_VENTA,TOTAL_VENDIDO
0,2542,ENVASE SADMAN TRANSP 105 ML PLATA CROMADO X 54...,59987.0,NaN,NaN,NaN,NaN,NaN
1,2699,ENVASE PM TRANSPARENTE GRIS AGRAFE 110 ML X 40...,400.0,NaN,NaN,NaN,NaN,NaN
18,0023,CILINDRO COLOR 30ML ROSA BABY X 180 UND,0.0,NaN,NaN,NaN,NaN,NaN
20,0026,CILINDRO COLOR 30ML AZUL BABY X 120 UND,0.0,NaN,NaN,NaN,NaN,NaN
27,0037,CILINDRO COLOR 30ML MORADO BABY X 120 UND,0.0,NaN,NaN,NaN,NaN,NaN
35,0047,CILINDRO COLOR 100 ML ROSA X 99 UND,0.0,NaN,NaN,NaN,NaN,NaN
36,0048,CILINDRO COLOR 100 ML VERDE X 99 UND,0.0,NaN,NaN,NaN,NaN,NaN
37,0049,CILINDRO TRANSP 2OZ AMARILLO X 112 UND,0.0,NaN,NaN,NaN,NaN,NaN
38,0050,CILINDRO TRANSP 2OZ AZUL CELESTE X 112 UND,0.0,NaN,NaN,NaN,NaN,NaN
39,0052,CILINDRO TRANSP 2OZ AZUL X 120 UND,0.0,NaN,NaN,NaN,NaN,NaN


In [312]:
# ============================================================
# EXPORTAR PRODUCTOS SIN VENTAS A EXCEL
# ============================================================

from pathlib import Path

columnas_productos_sin_ventas = [
    "CODIGO_PRODUCTO",
    "NOMBRE_PRODUCTO",
    "CANTIDAD_DISPONIBLE"
]

df_productos_sin_ventas[columnas_productos_sin_ventas].to_excel(
    ruta_productos_sin_ventas,
    index=False
)

print("Archivo de productos sin ventas generado correctamente:")
print(ruta_productos_sin_ventas)

Archivo de productos sin ventas generado correctamente:
..\Data\Productos_Sin_Ventas.xlsx


In [313]:
# ============================================================
# CONTINUAR ANÁLISIS SOLO CON PRODUCTOS QUE TIENEN VENTAS
# ============================================================

df_cobertura = df_productos_con_ventas.copy()

display(df_cobertura.head())
display(contar_nulos(df_cobertura))

,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CANTIDAD_DISPONIBLE,VENTA_PROMEDIO_MENSUAL,VENTA_MAXIMA_MENSUAL,VENTA_MINIMA_MENSUAL,MESES_CON_VENTA,TOTAL_VENDIDO
2,0001,ALCOHOL EXTRA NEUTRO.,2240.0,24178.275862,31825.0,7438.0,29.0,701170.0
3,0005,CAJA MADERA,3.0,2.000000,2.0,2.0,1.0,2.0
4,0006,CAJA PARA CILINDRO 1OZ NEGRA,8106.0,2596.551724,5500.0,150.0,29.0,75300.0
5,0007,CAJA PARA CILINDRO 1OZ PLATA,8487.0,1630.629630,3301.0,450.0,27.0,44027.0
6,0008,CAJA PARA CILINDRO 1OZ DORADO,7193.0,2014.750000,3650.0,25.0,28.0,56413.0


,COLUMNA,CANTIDAD_NULL
0,CODIGO_PRODUCTO,0
1,NOMBRE_PRODUCTO,0
2,CANTIDAD_DISPONIBLE,0
3,VENTA_PROMEDIO_MENSUAL,0
4,VENTA_MAXIMA_MENSUAL,0
5,VENTA_MINIMA_MENSUAL,0
6,MESES_CON_VENTA,0
7,TOTAL_VENDIDO,0


In [314]:
# ============================================================
# CALCULAR MESES DE COBERTURA
# ============================================================

df_cobertura["VENTA_PROMEDIO_MENSUAL"] = df_cobertura["VENTA_PROMEDIO_MENSUAL"].fillna(0)

df_cobertura["MESES_COBERTURA"] = np.where(
    df_cobertura["VENTA_PROMEDIO_MENSUAL"] > 0,
    df_cobertura["CANTIDAD_DISPONIBLE"] / df_cobertura["VENTA_PROMEDIO_MENSUAL"],
    np.nan
)

df_cobertura["MESES_COBERTURA"] = df_cobertura["MESES_COBERTURA"].round(2)

display(df_cobertura.head())

,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CANTIDAD_DISPONIBLE,VENTA_PROMEDIO_MENSUAL,VENTA_MAXIMA_MENSUAL,VENTA_MINIMA_MENSUAL,MESES_CON_VENTA,TOTAL_VENDIDO,MESES_COBERTURA
2,0001,ALCOHOL EXTRA NEUTRO.,2240.0,24178.275862,31825.0,7438.0,29.0,701170.0,0.09
3,0005,CAJA MADERA,3.0,2.000000,2.0,2.0,1.0,2.0,1.50
4,0006,CAJA PARA CILINDRO 1OZ NEGRA,8106.0,2596.551724,5500.0,150.0,29.0,75300.0,3.12
5,0007,CAJA PARA CILINDRO 1OZ PLATA,8487.0,1630.629630,3301.0,450.0,27.0,44027.0,5.20
6,0008,CAJA PARA CILINDRO 1OZ DORADO,7193.0,2014.750000,3650.0,25.0,28.0,56413.0,3.57


In [315]:
# ============================================================
# CLASIFICAR ESTADO DEL INVENTARIO
# ============================================================

def clasificar_inventario(meses):
    if pd.isna(meses):
        return "SIN VENTAS HISTÓRICAS"
    elif meses <= 1:
        return "CRÍTICO"
    elif meses <= 2:
        return "BAJO"
    elif meses <= 4:
        return "NORMAL"
    elif meses <= 8:
        return "ALTO"
    else:
        return "SOBRESTOCK"

df_cobertura["ESTADO_INVENTARIO"] = df_cobertura["MESES_COBERTURA"].apply(clasificar_inventario)

display(df_cobertura.head())

,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CANTIDAD_DISPONIBLE,VENTA_PROMEDIO_MENSUAL,VENTA_MAXIMA_MENSUAL,VENTA_MINIMA_MENSUAL,MESES_CON_VENTA,TOTAL_VENDIDO,MESES_COBERTURA,ESTADO_INVENTARIO
2,0001,ALCOHOL EXTRA NEUTRO.,2240.0,24178.275862,31825.0,7438.0,29.0,701170.0,0.09,CRÍTICO
3,0005,CAJA MADERA,3.0,2.000000,2.0,2.0,1.0,2.0,1.50,BAJO
4,0006,CAJA PARA CILINDRO 1OZ NEGRA,8106.0,2596.551724,5500.0,150.0,29.0,75300.0,3.12,NORMAL
5,0007,CAJA PARA CILINDRO 1OZ PLATA,8487.0,1630.629630,3301.0,450.0,27.0,44027.0,5.20,ALTO
6,0008,CAJA PARA CILINDRO 1OZ DORADO,7193.0,2014.750000,3650.0,25.0,28.0,56413.0,3.57,NORMAL


In [316]:
# ============================================================
# ORDENAR RESULTADO FINAL
# ============================================================

df_cobertura = df_cobertura.sort_values(
    by=["ESTADO_INVENTARIO", "MESES_COBERTURA", "NOMBRE_PRODUCTO"],
    ascending=[True, True, True]
).reset_index(drop=True)

display(df_cobertura.head(20))

,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CANTIDAD_DISPONIBLE,VENTA_PROMEDIO_MENSUAL,VENTA_MAXIMA_MENSUAL,VENTA_MINIMA_MENSUAL,MESES_CON_VENTA,TOTAL_VENDIDO,MESES_COBERTURA,ESTADO_INVENTARIO
0,0528,PEREDA HM,170473.0,42483.344828,77840.0,2380.0,29.0,1232017.0,4.01,ALTO
1,0883,STCK. ESC. INSP. EN ALLURE SPORT MEN,2015.0,501.379310,1440.0,80.0,29.0,14540.0,4.02,ALTO
2,1007,STCK. ESC. INSP. EN EAU DE CARTIER HM,2622.0,649.310345,1600.0,40.0,29.0,18830.0,4.04,ALTO
3,0607,TEXAS HM,81126.0,20009.785714,64950.0,2100.0,28.0,560274.0,4.05,ALTO
4,2829,COLOMBIA NEW VIBES UNISEX,125100.0,30820.000000,49690.0,11950.0,2.0,61640.0,4.06,ALTO
5,0109,ENVASE CRTR 30 ML VERDE X 160 UND,3332.0,819.150000,4120.0,249.0,20.0,16383.0,4.07,ALTO
6,0626,AZURE PERFECTION DM,163877.0,40145.620690,62670.0,15830.0,29.0,1164223.0,4.08,ALTO
7,0411,BORAMA HM,131600.0,32226.785714,54330.0,14045.0,28.0,902350.0,4.08,ALTO
8,1584,CILINDRO COLOR 30ML MORADO X 120 UND,33043.0,8081.444444,23510.0,15.0,9.0,72733.0,4.09,ALTO
9,0280,CUPIDO DM,148050.0,36222.000000,74500.0,6815.0,29.0,1050438.0,4.09,ALTO


In [317]:
# ============================================================
# CARGAR INVENTARIO
# ============================================================

df_table_maestra = pd.read_excel(ruta_tabla_maestra)

print("Dimensiones de la tabla maestra:", df_table_maestra.shape)

display(df_table_maestra.head())

Dimensiones de la tabla maestra: (2502, 6)


,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,CLASIFICACION I,CLASIFICACION II,CLASIFICACION III,CLASIFICACION IV
0,2542,ENVASE SADMAN TRANSP 105 ML PLATA CROMADO X 54...,ENVASE,SADMAN,105 ML,54
1,2610,ENVASE SALVAJE 100 ML FUCSIA X 72 UND,ENVASE,SALVAJE,100 ML,72
2,2608,ENVASE SALVAJE 100 ML NEGRO X 72 UND,ENVASE,SALVAJE,100 ML,72
3,2229,180 FUNDAS SURTIDAS X 6 COLORES,FUNDA,PAQUETE,NO APLICA,NO APLICA
4,2278,A MILKSHAKE PLEASE DM,ESENCIA,ESENCIA NICHO,ARMAF,DM


In [318]:
# ============================================================
# UNIR COBERTURA DE INVENTARIO CON TABLA MAESTRA DE PRODUCTOS
# ============================================================

# Copias de seguridad para no modificar los dataframe originales
df_cobertura_union = df_cobertura.copy()
df_productos_union = df_table_maestra.copy()

In [319]:
df_cobertura_union["CODIGO_PRODUCTO"] = (
    df_cobertura_union["CODIGO_PRODUCTO"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", "", regex=True)
)

df_productos_union["CODIGO_PRODUCTO"] = (
    df_productos_union["CODIGO_PRODUCTO"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", "", regex=True)
)

In [320]:
# ============================================================
# SELECCIONAR SOLO COLUMNAS NECESARIAS DE LA TABLA MAESTRA
# ============================================================

columnas_maestra = [
    "CODIGO_PRODUCTO",
    "NOMBRE_PRODUCTO",
    "CLASIFICACION I",
    "CLASIFICACION II",
    "CLASIFICACION III",
    "CLASIFICACION IV"
]

df_productos_maestra = df_productos_union[columnas_maestra].copy()

# Renombrar nombre de producto de la tabla maestra
# para no confundirse con el nombre que viene del inventario
df_productos_maestra = df_productos_maestra.rename(
    columns={
        "NOMBRE_PRODUCTO": "NOMBRE_PRODUCTO_MAESTRO"
    }
)

In [321]:
# ============================================================
# UNIR COBERTURA CON TABLA MAESTRA
# ============================================================

df_cobertura_final = df_cobertura_union.merge(
    df_productos_maestra,
    on="CODIGO_PRODUCTO",
    how="left"
)

In [322]:
# ============================================================
# ORDENAR COLUMNAS FINALES
# ============================================================

columnas_ordenadas = [
    "CODIGO_PRODUCTO",
    "NOMBRE_PRODUCTO",
    "NOMBRE_PRODUCTO_MAESTRO",
    "CLASIFICACION I",
    "CLASIFICACION II",
    "CLASIFICACION III",
    "CLASIFICACION IV",
    "CANTIDAD_DISPONIBLE",
    "VENTA_PROMEDIO_MENSUAL",
    "VENTA_MAXIMA_MENSUAL",
    "VENTA_MINIMA_MENSUAL",
    "MESES_CON_VENTA",
    "TOTAL_VENDIDO",
    "MESES_COBERTURA",
    "ESTADO_INVENTARIO"
]

df_cobertura_final = df_cobertura_final[columnas_ordenadas]

In [323]:
# ============================================================
# VISUALIZAR RESULTADO
# ============================================================

display(df_cobertura_final.head(20))

,CODIGO_PRODUCTO,NOMBRE_PRODUCTO,NOMBRE_PRODUCTO_MAESTRO,CLASIFICACION I,CLASIFICACION II,CLASIFICACION III,CLASIFICACION IV,CANTIDAD_DISPONIBLE,VENTA_PROMEDIO_MENSUAL,VENTA_MAXIMA_MENSUAL,VENTA_MINIMA_MENSUAL,MESES_CON_VENTA,TOTAL_VENDIDO,MESES_COBERTURA,ESTADO_INVENTARIO
0,0528,PEREDA HM,PEREDA HM,ESENCIA,ESENCIA GENERAL,PERRY ELLIS,HM,170473.0,42483.344828,77840.0,2380.0,29.0,1232017.0,4.01,ALTO
1,0883,STCK. ESC. INSP. EN ALLURE SPORT MEN,STCK. ESC. INSP. EN ALLURE SPORT MEN,STCK,NO APLICA,NO APLICA,NO APLICA,2015.0,501.379310,1440.0,80.0,29.0,14540.0,4.02,ALTO
2,1007,STCK. ESC. INSP. EN EAU DE CARTIER HM,STCK. ESC. INSP. EN EAU DE CARTIER HM,STCK,NO APLICA,NO APLICA,NO APLICA,2622.0,649.310345,1600.0,40.0,29.0,18830.0,4.04,ALTO
3,0607,TEXAS HM,TEXAS HM,ESENCIA,ESENCIA GENERAL,VERSACE,HM,81126.0,20009.785714,64950.0,2100.0,28.0,560274.0,4.05,ALTO
4,2829,COLOMBIA NEW VIBES UNISEX,COLOMBIA NEW VIBES UNISEX,ESENCIA,ESENCIA NICHO,LORENZO PAZZAGLIA,UNISEX,125100.0,30820.000000,49690.0,11950.0,2.0,61640.0,4.06,ALTO
5,0109,ENVASE CRTR 30 ML VERDE X 160 UND,ENVASE CRTR 30 ML VERDE X 160 UND,ENVASE,CRTR,30 ML,160,3332.0,819.150000,4120.0,249.0,20.0,16383.0,4.07,ALTO
6,0626,AZURE PERFECTION DM,AZURE PERFECTION DM,ESENCIA,ESENCIA GENERAL,YANBAL,DM,163877.0,40145.620690,62670.0,15830.0,29.0,1164223.0,4.08,ALTO
7,0411,BORAMA HM,BORAMA HM,ESENCIA,ESENCIA GENERAL,HUGO BOSS,HM,131600.0,32226.785714,54330.0,14045.0,28.0,902350.0,4.08,ALTO
8,1584,CILINDRO COLOR 30ML MORADO X 120 UND,CILINDRO COLOR 30ML MORADO X 120 UND,ENVASE,CILINDRO,30 ML,120,33043.0,8081.444444,23510.0,15.0,9.0,72733.0,4.09,ALTO
9,0280,CUPIDO DM,CUPIDO DM,ESENCIA,ESENCIA GENERAL,CACHAREL,DM,148050.0,36222.000000,74500.0,6815.0,29.0,1050438.0,4.09,ALTO


In [325]:
# ============================================================
# EXPORTAR ANÁLISIS DE INVENTARIO
# ============================================================

with pd.ExcelWriter(ruta_salida_inventario, engine="openpyxl") as writer:
    
#    df_inventario.to_excel(
#        writer,
#        sheet_name="Inventario_Limpio",
#        index=False
#    )
    
#    df_promedio_ventas.to_excel(
#        writer,
#        sheet_name="Promedio_Ventas",
#        index=False
#    )
    
    df_cobertura_final.to_excel(
        writer,
        sheet_name="Cobertura_Inventario",
        index=False
    )

print("Archivo generado correctamente:")
print(ruta_salida_inventario)

Archivo generado correctamente:
..\Resultado\Analisis_Inventario_Cobertura.xlsx


In [326]:
# ============================================================
# RESUMEN GENERAL DEL INVENTARIO
# ============================================================

resumen_estado = (
    df_cobertura_final
    .groupby("ESTADO_INVENTARIO")
    .agg(
        PRODUCTOS=("CODIGO_PRODUCTO", "count"),
        INVENTARIO_TOTAL=("CANTIDAD_DISPONIBLE", "sum")
    )
    .reset_index()
)

display(resumen_estado)

,ESTADO_INVENTARIO,PRODUCTOS,INVENTARIO_TOTAL
0,ALTO,419,62418781.0
1,BAJO,135,3852719.0
2,CRÍTICO,577,660518.0
3,NORMAL,289,28276786.0
4,SOBRESTOCK,1047,47004321.0
